In [1]:
"""
PASSO 5 — Eva_Performance: Backtesting e calculo das 8 metricas.

VERSAO HIBRIDA: usa o melhor dataset para cada modelo:
    - MI features:  NB, LR, MLP, RNN, XGB
    - 44 features:  SAE, SVM, CART, RF, LSTM, GRU, DBN

Para modelos MI: le {Code}_TradeSignal_{Algo}_MI.csv de B3ICSMI
Para modelos 44: le {Code}_TradeSignal_{Algo}.csv de B3ICS

Os 4 cenarios:
    - In_InS:  Sample="In",  Start="2018-01-02", End="2019-12-31"
    - In_OutS: Sample="In",  Start="2020-01-02", End="2020-06-30"
    - Out_InS: Sample="Out", Start="2018-01-02", End="2019-12-31"
    - Out_OutS:Sample="Out", Start="2020-01-02", End="2020-06-30"

Uso:
    python 05_eva_performance.py
"""

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ===================== CONFIGURACAO =====================
BASE_DIR = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS")
BASE_MI  = Path(r"C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICSMI")
SEC_NAMES = BASE_DIR / ".NewB3_pruned.csv"
INDEX_FILE = BASE_DIR / ".IBOV_Index.csv"

# Modelos que usam MI features (melhor desempenho com selecao)
MI_MODELS = {"NB", "LR", "MLP", "RNN", "XGB"}

# Modelos que usam 44 features (melhor desempenho sem selecao)
FULL_MODELS = {"SAE", "SVM", "CART", "RF", "LSTM", "GRU", "DBN"}

ALGORITHMS = sorted(MI_MODELS | FULL_MODELS)

SCENARIOS = {
    "In_InS":  {"Sample": "In",  "Start": "2018-01-02", "End": "2019-12-31"},
    "In_OutS": {"Sample": "In",  "Start": "2020-01-02", "End": "2020-06-30"},
    "Out_InS": {"Sample": "Out", "Start": "2018-01-02", "End": "2019-12-31"},
    "Out_OutS":{"Sample": "Out", "Start": "2020-01-02", "End": "2020-06-30"},
}

ANNUALIZATION = 250
# ========================================================


def get_signal_path(code, algorithm, base_dir, base_mi):
    if algorithm in MI_MODELS:
        return base_mi / f"{code}_TradeSignal_{algorithm}_MI.csv"
    else:
        return base_dir / f"{code}_TradeSignal_{algorithm}.csv"


# ============================================================
#              8 METRICAS DE PERFORMANCE
# ============================================================

def win_rate(returns):
    nonzero = returns[returns != 0]
    if len(nonzero) == 0:
        return 0.0
    return len(nonzero[nonzero > 0]) / len(nonzero)

def annualized_return(returns, scale=ANNUALIZATION):
    return returns.mean() * scale

def annualized_sharpe(returns, scale=ANNUALIZATION, rf=0.0):
    mean_ret = returns.mean()
    std_ret = returns.std(ddof=1)
    if std_ret == 0 or np.isnan(std_ret):
        return 0.0
    return np.sqrt(scale) * (mean_ret - rf / scale) / std_ret

def max_drawdown(returns):
    cumret = (1 + returns).cumprod()
    running_max = cumret.cummax()
    drawdowns = (cumret - running_max) / running_max
    return abs(drawdowns.min())

def calmar_ratio(returns, scale=ANNUALIZATION):
    arr = annualized_return(returns, scale)
    mdd = max_drawdown(returns)
    if mdd == 0 or np.isnan(mdd):
        return 0.0
    return arr / mdd

def sortino_ratio(returns, mar=0.0):
    downside = returns[returns < mar]
    if len(downside) == 0:
        return 0.0
    downside_std = np.sqrt((downside ** 2).mean())
    if downside_std == 0:
        return 0.0
    return (returns.mean() - mar) / downside_std

def adjusted_sharpe_ratio(returns, scale=ANNUALIZATION):
    asr = annualized_sharpe(returns, scale)
    n = len(returns)
    if n < 4:
        return asr
    mean_r = returns.mean()
    std_r = returns.std(ddof=1)
    if std_r == 0:
        return 0.0
    skew = ((returns - mean_r) ** 3).mean() / (std_r ** 3)
    kurt = ((returns - mean_r) ** 4).mean() / (std_r ** 4) - 3
    return asr * (1 + (skew / 6) * asr - (kurt / 24) * asr ** 2)

def average_recovery(returns):
    cumret = (1 + returns).cumprod()
    running_max = cumret.cummax()
    in_drawdown = cumret < running_max
    recovery_periods = []
    current_length = 0
    for i in range(len(in_drawdown)):
        if in_drawdown.iloc[i]:
            current_length += 1
        else:
            if current_length > 0:
                recovery_periods.append(current_length)
            current_length = 0
    if current_length > 0:
        recovery_periods.append(current_length)
    if len(recovery_periods) == 0:
        return 0.0
    return np.mean(recovery_periods)

def compute_performance(returns):
    returns = returns.dropna()
    if len(returns) < 10:
        return [np.nan] * 8
    return [
        win_rate(returns),
        annualized_return(returns),
        annualized_sharpe(returns),
        max_drawdown(returns),
        calmar_ratio(returns),
        sortino_ratio(returns),
        adjusted_sharpe_ratio(returns),
        average_recovery(returns),
    ]

METRIC_NAMES = ["WR", "ARR", "ASR", "MDD", "CAR", "SOR", "MSR", "ART"]
STRATEGY_NAMES = ["Index", "BAH", "Trade"]
COLUMN_NAMES = [f"{m}{s}" for m in METRIC_NAMES for s in STRATEGY_NAMES]


# ============================================================
#              DOWNLOAD DO IBOVESPA
# ============================================================

def ensure_index_file(index_path):
    if index_path.exists():
        return True
    print("[INFO] Baixando indice Ibovespa (^BVSP) via yfinance...")
    try:
        import yfinance as yf
        raw = yf.download("^BVSP", start="2016-01-01", end="2020-07-01",
                          progress=False, auto_adjust=False)
        if raw is None or raw.empty:
            return False
        if isinstance(raw.columns, pd.MultiIndex):
            raw.columns = raw.columns.get_level_values(0)
        factor = raw["Adj Close"] / raw["Close"]
        factor = factor.replace([np.inf, -np.inf], np.nan).ffill().fillna(1.0)
        out = pd.DataFrame()
        out["Date"] = raw.index
        out["Open"] = (raw["Open"] * factor).values
        out["High"] = (raw["High"] * factor).values
        out["Low"] = (raw["Low"] * factor).values
        out["Close"] = raw["Adj Close"].values
        out["Volume"] = raw["Volume"].values
        out["Date"] = pd.to_datetime(out["Date"]).dt.strftime("%Y-%m-%d")
        out.to_csv(index_path, index=False, encoding="utf-8-sig")
        return True
    except Exception as e:
        print(f"[ERRO] {e}")
        return False


# ============================================================
#              FUNCAO PRINCIPAL
# ============================================================

def load_index(index_path):
    df = pd.read_csv(index_path, encoding="utf-8-sig")
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).sort_values(date_col).set_index(date_col)
    close = df["Close"].astype(float)
    ret = close.pct_change().dropna()
    ret.name = "IndexRet"
    return ret


def eva_func(sec_names_path, base_dir, base_mi, index_ret,
             algorithm, industry, sample, start, end, scenario_label):
    codes_df = pd.read_csv(sec_names_path, dtype=str, encoding="utf-8-sig")
    codes_df.columns = [c.strip() for c in codes_df.columns]
    mask = (codes_df["industry"].str.strip() == industry) & \
           (codes_df["Sample"].str.strip() == sample)
    codes = codes_df.loc[mask, "Code"].str.strip().str.upper().tolist()

    if not codes:
        return None

    start_dt = pd.to_datetime(start)
    end_dt = pd.to_datetime(end)
    results = []

    for code in codes:
        signal_file = get_signal_path(code, algorithm, base_dir, base_mi)
        raw_file = base_dir / f"{code}_New.csv"

        if not signal_file.exists() or not raw_file.exists():
            continue

        try:
            sig_df = pd.read_csv(signal_file, encoding="utf-8-sig")
            sig_df.columns = [c.strip() for c in sig_df.columns]
            date_col_sig = sig_df.columns[0]
            sig_df[date_col_sig] = pd.to_datetime(sig_df[date_col_sig], errors="coerce")
            sig_df = sig_df.dropna(subset=[date_col_sig])
            signal = pd.Series(sig_df.iloc[:, 1].values,
                               index=sig_df[date_col_sig].values).sort_index()
            signal.index = pd.to_datetime(signal.index)

            signal = signal[(signal.index >= start_dt) & (signal.index <= end_dt)]
            if len(signal) < 10:
                continue

            signal = signal.shift(1)

            raw_df = pd.read_csv(raw_file, encoding="utf-8-sig")
            raw_df.columns = [c.strip() for c in raw_df.columns]
            date_col_raw = raw_df.columns[0]
            raw_df[date_col_raw] = pd.to_datetime(raw_df[date_col_raw], errors="coerce")
            raw_df = raw_df.dropna(subset=[date_col_raw]).sort_values(date_col_raw)
            raw_df = raw_df.set_index(date_col_raw)
            close = raw_df["Close"].astype(float)
            raw_ret = close.pct_change()

            common_dates = signal.dropna().index.intersection(raw_ret.dropna().index)
            common_dates = common_dates.intersection(index_ret.dropna().index)

            if len(common_dates) < 10:
                continue

            sig_aligned = signal.loc[common_dates].astype(float)
            raw_aligned = raw_ret.loc[common_dates].astype(float)
            idx_aligned = index_ret.loc[common_dates].astype(float)

            trade_ret = sig_aligned * raw_aligned

            perf_index = compute_performance(idx_aligned)
            perf_bah = compute_performance(raw_aligned)
            perf_trade = compute_performance(trade_ret)

            row = []
            for m in range(8):
                row.extend([perf_index[m], perf_bah[m], perf_trade[m]])

            results.append({"Code": code, **{COLUMN_NAMES[j]: row[j] for j in range(len(row))}})

        except Exception:
            continue

    if not results:
        return None

    result_df = pd.DataFrame(results).set_index("Code")
    outfile = base_dir / f".Performance_{algorithm}_{industry}_{sample}_{scenario_label}.csv"
    result_df.to_csv(outfile, encoding="utf-8-sig")
    return result_df


def main():
    if not ensure_index_file(INDEX_FILE):
        print("[ERRO] Nao foi possivel obter o indice Ibovespa.")
        return

    index_ret = load_index(INDEX_FILE)

    codes_df = pd.read_csv(SEC_NAMES, dtype=str, encoding="utf-8-sig")
    codes_df.columns = [c.strip() for c in codes_df.columns]
    industries = sorted(codes_df["industry"].str.strip().unique())

    print(f"{'='*60}")
    print(f"PASSO 5: Eva_Performance (versao hibrida)")
    print(f"{'='*60}")
    print(f"Modelos com MI features:  {sorted(MI_MODELS)}")
    print(f"Modelos com 44 features:  {sorted(FULL_MODELS)}")
    print(f"Diretorio MI:  {BASE_MI}")
    print(f"Diretorio 44:  {BASE_DIR}")
    print(f"Algoritmos: {ALGORITHMS}")
    print(f"Setores: {industries}")
    print(f"Cenarios: {list(SCENARIOS.keys())}")
    total = len(ALGORITHMS) * len(industries) * len(SCENARIOS)
    print(f"Total de combinacoes: {total}\n")

    summary = []

    for scen_name, scen_params in SCENARIOS.items():
        sample = scen_params["Sample"]
        start = scen_params["Start"]
        end = scen_params["End"]

        for algo in tqdm(ALGORITHMS, desc=f"Cenario {scen_name}"):
            feat_type = "MI" if algo in MI_MODELS else "44"
            for ind in industries:
                result = eva_func(
                    SEC_NAMES, BASE_DIR, BASE_MI, index_ret,
                    algorithm=algo,
                    industry=ind,
                    sample=sample,
                    start=start,
                    end=end,
                    scenario_label=scen_name,
                )

                if result is not None and len(result) > 0:
                    means = result.mean(axis=0)
                    summary.append({
                        "Scenario": scen_name,
                        "Algorithm": algo,
                        "Industry": ind,
                        "Features": feat_type,
                        "n_stocks": len(result),
                        **{col: means[col] for col in COLUMN_NAMES if col in means.index}
                    })

    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(BASE_DIR / "eva_performance_summary.csv",
                      index=False, encoding="utf-8-sig")

    for scen_name in SCENARIOS.keys():
        scen_data = summary_df[summary_df["Scenario"] == scen_name]
        if scen_data.empty:
            continue

        for metric in METRIC_NAMES:
            trade_col = f"{metric}Trade"
            if trade_col not in scen_data.columns:
                continue

            pivot = scen_data.pivot_table(
                index="Algorithm", columns="Industry",
                values=trade_col, aggfunc="mean"
            )
            outfile = BASE_DIR / f".indi_{metric}_{scen_name}.csv"
            pivot.to_csv(outfile, encoding="utf-8-sig")

    print(f"\n{'='*60}")
    print(f"Concluido!")
    print(f"Resumo: eva_performance_summary.csv")
    print(f"\nConfiguracao por modelo:")
    for algo in ALGORITHMS:
        feat = "MI features" if algo in MI_MODELS else "44 features"
        print(f"  {algo:>5}: {feat}")


if __name__ == "__main__":
    main()

PASSO 5: Eva_Performance (versao hibrida)
Modelos com MI features:  ['LR', 'MLP', 'NB', 'RNN', 'XGB']
Modelos com 44 features:  ['CART', 'DBN', 'GRU', 'LSTM', 'RF', 'SAE', 'SVM']
Diretorio MI:  C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICSMI
Diretorio 44:  C:\Users\Rodrigo\Desktop\Replicando para B3_2\B3ICS
Algoritmos: ['CART', 'DBN', 'GRU', 'LR', 'LSTM', 'MLP', 'NB', 'RF', 'RNN', 'SAE', 'SVM', 'XGB']
Setores: ['BM', 'CC', 'CNC', 'COM', 'FIN', 'GCS', 'OGB', 'UTI']
Cenarios: ['In_InS', 'In_OutS', 'Out_InS', 'Out_OutS']
Total de combinacoes: 384



Cenario Out_OutS: 100%|██████████| 12/12 [00:24<00:00,  2.01s/it]



Concluido!
Resumo: eva_performance_summary.csv

Configuracao por modelo:
   CART: 44 features
    DBN: 44 features
    GRU: 44 features
     LR: MI features
   LSTM: 44 features
    MLP: MI features
     NB: MI features
     RF: 44 features
    RNN: MI features
    SAE: 44 features
    SVM: 44 features
    XGB: MI features
